# SmolLM2 V11 — Multi-Layer Activation-Aware Ternary CeNN with Quantization-Aware Distillation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/SmolLM2_360M_V11_MultiLayer_Ternary_CeNN_QAT_Distillation_Colab.ipynb)

**A real multi-layer training experiment on** [SmolLM2-360M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct), a causal decoder. Based on the **actual V10 evidence**: only `model.layers.28.self_attn.o_proj` survived one-shot ternary conversion, saving 1.509 MiB (0.193% of the whole model), 100% teacher agreement on just 12 last-token validation decisions, test PPL 34.879 vs original 34.764; most measured inference shapes ran slower. The untrained direct CeNN-only replacement got 6/8 agreement. V11 does **not** claim all-layer feasibility or a speedup before testing.

### Research design

1. **Activation-aware screening:** sample true SmolLM2 teacher inputs/outputs from **all 32** attention `o_proj` layers plus the last 8 MLP `down_proj` layers. Rank their *local* G128 ternary reconstruction MSE; explicitly include V10-successful layer 28.
2. **Ternary QAT:** learn FP32 master ternary codes with a straight-through estimator and FP32 log-G128-scales against *teacher projection activations*. Optional learnable, causal-safe cellular feature stencil `[-1,+1,0]` (acts across hidden channels, not future tokens); no concatenation.
3. **End-to-end teacher distillation:** a small number of gradient updates to the candidate projection only, optimizing last-16-token full-vocabulary KL plus gold-token cross entropy while the other model weights remain frozen. If strict ternary fails, optionally try a rank-16 FP16 error-residual *clearly labeled as hybrid*.
4. **Strict checkpoints:** only training-row calibration for gradients; disjoint train-row DEV/GUARD for acceptance; WikiText **validation and test** used only once after all layer choices are frozen. Preserve all 32 original attention layers except accepted replacements.
5. **Reality checks:** final validation/test teacher agreement across hundreds of *token positions*, gold-label perplexity, token generation samples, actual state size, paired CUDA latency and a cold-load parity test for exported packed 2-bit checkpoint delta.

**Fast T4 defaults:** screen 40 modules in one teacher forward; train at most 6 candidates for 12 local + 4 global steps (per profile), accept at most 4, evaluate only 2 chat prompts. Increase budgets **after** the first results ZIP. No custom Triton compilation, no MSLK, no bitsandbytes. This remains an experimental **packed 2-bit storage** runtime that decodes FP16 output tiles; no native ternary Tensor Core speedup is claimed.

**Source checkpoint revision** pinned to V10: `a10cc1512eabd3dde888204e902eca88bddb4951`. Official held-out test is never used to choose layer replacements. Nothing is uploaded to Hugging Face automatically.


In [ ]:
#@title 1. Lightweight binary dependency setup (keep Colab torch and CUDA)
import os,sys,json,math,gc,zipfile,copy,random,subprocess,importlib.util,statistics
from pathlib import Path
from urllib.request import urlopen
import numpy as np,pandas as pd,torch
from torch import nn
from torch.nn import functional as F
from IPython.display import display
missing=[]
for mod,pkg in [
    ("transformers","transformers>=4.50,<5"),
    ("datasets","datasets>=2.20,<5"),
    ("safetensors","safetensors>=0.4"),
    ("huggingface_hub","huggingface_hub>=0.35"),
    ("pandas","pandas")]:
    if importlib.util.find_spec(mod) is None:missing.append(pkg)
if missing:
    subprocess.run([sys.executable,"-m","pip","install","-q",
        "--only-binary=:all:",*missing],check=True)
from transformers import AutoModelForCausalLM,AutoTokenizer
from huggingface_hub import HfApi
from safetensors.torch import save_file,load_file
assert torch.cuda.is_available(),"Select Tesla T4, L4, or better GPU in Colab"
print("PyTorch",torch.__version__,"GPU",torch.cuda.get_device_name(0))


In [ ]:
#@title 2. Quality-first T4 controls, exact revisions, and 2-bit QAT runtime
SOURCE="HuggingFaceTB/SmolLM2-360M-Instruct" #@param {type:"string"}
SOURCE_SHA="a10cc1512eabd3dde888204e902eca88bddb4951" #@param {type:"string"}
ALL_ATTENTION=True #@param {type:"boolean"}
LAST_MLP=8 #@param {type:"integer"}
MAX_TRAIN_CANDIDATES=6 #@param {type:"integer"}
MAX_ACCEPTED=4 #@param {type:"integer"}
LOCAL_STEPS=12 #@param {type:"integer"}
GLOBAL_DISTILL_STEPS=4 #@param {type:"integer"}
USE_RANK16_RESCUE=True #@param {type:"boolean"}
USE_FEATURE_CENN=True #@param {type:"boolean"}
CHUNK_ROWS=128 #@param {type:"integer"}
TRAIN_ROWS_PER_PROJECTION=96 #@param {type:"integer"}
CAL_WINDOWS=12 #@param {type:"integer"}
DEV_WINDOWS=24 #@param {type:"integer"}
GUARD_WINDOWS=24 #@param {type:"integer"}
AUDIT_WINDOWS=48 #@param {type:"integer"}
TEST_WINDOWS=48 #@param {type:"integer"}
SEQ_LEN=96 #@param {type:"integer"}
DECISION_POSITIONS=16 #@param {type:"integer"}
MAX_DISAGREEMENTS=4 #@param {type:"integer"}
MAX_JS=0.004 #@param {type:"number"}
MAX_DEV_PPL_INCREASE=0.015 #@param {type:"number"}
TRAIN_LOCAL_LR=0.008 #@param {type:"number"}
TRAIN_GLOBAL_LR=0.001 #@param {type:"number"}
BENCH_REPEATS=5 #@param {type:"integer"}
SEED=2071
OUT=Path("/content/SmolLM2_360M_V11_Ternary_CeNN_QAT")
OUT.mkdir(parents=True,exist_ok=True)
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
assert SEQ_LEN>=DECISION_POSITIONS>=1 and LAST_MLP>=0
assert MAX_TRAIN_CANDIDATES>=0 and 0<=MAX_ACCEPTED<=MAX_TRAIN_CANDIDATES
assert TRAIN_ROWS_PER_PROJECTION>0 and CHUNK_ROWS>0
try:
    from google.colab import userdata
    HF_TOKEN=userdata.get("HF_TOKEN")
except Exception:HF_TOKEN=os.environ.get("HF_TOKEN")
def remote_module(sha,file,name):
    target=Path("/content")/file
    with urlopen(f"https://raw.githubusercontent.com/vtavakkoli/TinyCeNN-LM/{sha}/src/tinycenn_lm/{file}",timeout=90) as response:
        target.write_bytes(response.read())
    spec=importlib.util.spec_from_file_location(name,str(target))
    module=importlib.util.module_from_spec(spec)
    sys.modules[name]=module
    spec.loader.exec_module(module)
    return module
V9_SHA="4ea3bf6e5001b7b123d7c137cb09f0ce32cc1c55"
V11_SHA="6d8cf30b127b0383617b04e442664fb21fc1daf7"
v9=remote_module(V9_SHA,"laya_cenn_ternary_concat_v9.py","laya_cenn_ternary_concat_v9")
qat=remote_module(V11_SHA,"smol_ternary_cenn_v11.py","smol_ternary_cenn_v11")
tokenizer=AutoTokenizer.from_pretrained(SOURCE,revision=SOURCE_SHA,token=HF_TOKEN)
if tokenizer.pad_token_id is None:tokenizer.pad_token=tokenizer.eos_token
teacher=AutoModelForCausalLM.from_pretrained(
    SOURCE,revision=SOURCE_SHA,token=HF_TOKEN,
    torch_dtype=torch.float16,attn_implementation="sdpa").cuda().eval().requires_grad_(False)
N_LAYERS=len(teacher.model.layers)
assert N_LAYERS==32,f"Expected 32 SmolLM2 layers, got {N_LAYERS}"
NAMES=qat.candidate_paths(teacher,mlp_last=LAST_MLP,attention_all=ALL_ATTENTION)
assert len(NAMES)>=1
print("Teacher source",SOURCE,SOURCE_SHA,"layers",N_LAYERS,
      "eligible projection sites",len(NAMES))
print("Original FP16 model state MiB",qat.tensor_storage_bytes(teacher)/2**20)


In [ ]:
#@title 3. CPU/GPU smoke tests: STE gradients, pure ternary export, feature CeNN causal safety
linear=nn.Linear(256,128,device="cuda",dtype=torch.float16)
input_probe=torch.randn(2,8,256,device="cuda",dtype=torch.float16)
train_module=qat.TernaryQATLinear(linear,rank=0,feature_residual=True).cuda()
with torch.enable_grad():
    sample=train_module(input_probe)
    error=(sample.float()-linear(input_probe).float()).square().mean()
    error.backward()
assert train_module.codes.grad is not None
assert bool(torch.isfinite(train_module.codes.grad).all())
assert train_module.log_scales.grad is not None
    # Their gradients must reach both QAT code and G128 scale.
packed=train_module.pack(chunk_rows=64).cuda().eval()
with torch.inference_mode():
    forward_packed=packed(input_probe)
    forward_fake=train_module(input_probe)
torch.testing.assert_close(forward_fake,forward_packed,rtol=.04,atol=.06)
assert packed.base.packed_weight.dtype==torch.uint8
assert not hasattr(packed.base,"_cached")
print("QAT gradients, packed 2-bit parity, no-cache and feature CeNN smoke checks PASS")
del linear,input_probe,train_module,packed,forward_packed,forward_fake
gc.collect();torch.cuda.empty_cache()


In [ ]:
#@title 4. Source-case-disjoint WikiText train CAL/DEV/GUARD; held-out validation/test
from datasets import load_dataset
WIKI_REPO="Salesforce/wikitext"
WIKI_CONFIG="wikitext-2-raw-v1"
wiki=load_dataset(WIKI_REPO,WIKI_CONFIG)
assert all(x in wiki for x in ("train","validation","test"))
def windows_from(rows,count,length=SEQ_LEN):
    stream=[]
    for row in rows:
        text=row["text"].strip()
        if len(text)<70:continue
        stream+=tokenizer.encode(text,add_special_tokens=False)+[tokenizer.eos_token_id]
        if len(stream)>=(count+1)*(length+1):break
    result=[stream[j*(length+1):(j+1)*(length+1)] for j in range(count)]
    if len(result)!=count or any(len(w)!=length+1 for w in result):
        raise ValueError(f"Not enough WikiText tokens for {count} windows")
    return result
train=wiki["train"]
CAL_TEXT=windows_from(train.select(range(0,min(600,len(train)))),CAL_WINDOWS)
DEV_TEXT=windows_from(train.select(range(600,min(1200,len(train)))),DEV_WINDOWS)
GUARD_TEXT=windows_from(train.select(range(1200,min(1800,len(train)))),GUARD_WINDOWS)
AUDIT_TEXT=windows_from(wiki["validation"],AUDIT_WINDOWS)
TEST_TEXT=windows_from(wiki["test"],TEST_WINDOWS)
def batches(windows,size=2,labels=False):
    out=[]
    for idx in range(0,len(windows),size):
        seqs=torch.tensor(windows[idx:idx+size],device="cuda",dtype=torch.long)
        ids=seqs[:,:-1]
        inputs={"input_ids":ids,"attention_mask":torch.ones_like(ids)}
        out.append((inputs,seqs[:,1:]) if labels else inputs)
    return out
CAL_PAIRS=batches(CAL_TEXT,labels=True)
CAL=[x for x,_ in CAL_PAIRS]
DEV=batches(DEV_TEXT)
GUARD=batches(GUARD_TEXT)
AUDIT=batches(AUDIT_TEXT)
TEST=batches(TEST_TEXT)
@torch.inference_mode()
def ref_for(part):
    return [teacher(**b,use_cache=False).logits[:,-DECISION_POSITIONS:,:].float().cpu()
            for b in part]
REF_DEV=ref_for(DEV)
REF_GUARD=ref_for(GUARD)
REF_CAL=ref_for(CAL)
DEV_BASE_PPL=qat.perplexity(teacher,DEV_TEXT)["perplexity"]
print("Disjoint cases:",{k:len(v) for k,v in [
  ("calibration",CAL_TEXT),("development",DEV_TEXT),("guard",GUARD_TEXT),
  ("validation",AUDIT_TEXT),("test",TEST_TEXT)]})
print("Baseline DEV gold PPL",DEV_BASE_PPL)


In [ ]:
#@title 5. Screen all candidate attention/MLP sites using REAL teacher activations
# One teacher forward pass collects per-site activations; no future text leaks.
SAMPLES={}
handles=[]
def add_hook(name):
    def hook(module,inputs,output):
        inp=inputs[0].detach().reshape(-1,inputs[0].shape[-1])
        y=output[0] if isinstance(output,(tuple,list)) else output
        yy=y.detach().reshape(-1,y.shape[-1])
        take=min(TRAIN_ROWS_PER_PROJECTION,len(inp))
        SAMPLES[name]=(inp[:take].half().cpu(),yy[:take].half().cpu())
    return hook
for name in NAMES:
    handles.append(teacher.get_submodule(name).register_forward_hook(add_hook(name)))
try:
    with torch.inference_mode():teacher(**CAL[0],use_cache=False)
finally:
    for handle in handles:handle.remove()
assert len(SAMPLES)==len(NAMES),f"Missing activations for {set(NAMES)-set(SAMPLES)}"
SCREEN=[]
for name in NAMES:
    linear=teacher.get_submodule(name)
    x,y=(t.cuda() for t in SAMPLES[name])
    baseline=qat.TernaryQATLinear(
        linear,rank=0,feature_residual=False)
    with torch.inference_mode():
        yq=baseline(x).float()
        rel=float((yq-y.float()).square().mean()/
                  y.float().square().mean().clamp_min(1e-10))
    SCREEN.append(dict(name=name,local_relative_mse=rel,
        source_projection_bytes=qat.projection_bytes(linear)))
    del baseline,x,y,yq
    if len(SCREEN)%8==0:print("Screened",len(SCREEN),"/",len(NAMES))
screen_df=pd.DataFrame(SCREEN).sort_values("local_relative_mse")
screen_df.to_csv(OUT/"all32_attention_ternary_activation_screen.csv",index=False)
display(screen_df.head(18))
o_rank=sorted((r for r in SCREEN if r["name"].endswith("self_attn.o_proj")),
              key=lambda r:r["local_relative_mse"])
m_rank=sorted((r for r in SCREEN if r["name"].endswith("mlp.down_proj")),
              key=lambda r:r["local_relative_mse"])
seed="model.layers.28.self_attn.o_proj"
ordered=[seed] if seed in NAMES else []
while o_rank or m_rank:
    for _ in range(2):
        if o_rank:ordered.append(o_rank.pop(0)["name"])
    if m_rank:ordered.append(m_rank.pop(0)["name"])
ordered=list(dict.fromkeys(ordered))
TRAIN_ORDER=ordered[:MAX_TRAIN_CANDIDATES]
(OUT/"training_order.json").write_text(json.dumps({
    "names":TRAIN_ORDER,"screened_sites":len(NAMES),
    "v10_seed":seed,"policy":"seed layer 28 then 2 ranked attention to 1 ranked MLP"},indent=2))
print("Training candidates:",TRAIN_ORDER)


In [ ]:
#@title 6. Activation-aware local QAT and full-model KL teacher distillation
student=copy.deepcopy(teacher).eval().requires_grad_(False)
ORIGINAL_STATE=qat.tensor_storage_bytes(teacher)
SELECTED=[]
TRIALS=[]
def optimize_candidate(module,name,rank=0):
    xx,yy=SAMPLES[name]
    xx=xx.to("cuda");yy=yy.to("cuda")
    # Local activation-output reconstruction; gradients only candidate codes/scales.
    optim=torch.optim.AdamW(module.parameters(),lr=TRAIN_LOCAL_LR,weight_decay=0)
    local_history=[]
    module.train()
    for step in range(LOCAL_STEPS):
        if len(xx)>64:
            idx=torch.randperm(len(xx),device="cuda")[:64]
            x,y=xx[idx],yy[idx]
        else:x,y=xx,yy
        optim.zero_grad(set_to_none=True)
        prediction=module(x).float()
        loss=((prediction-y.float()).square().mean() /
              y.float().square().mean().clamp_min(1.e-8))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(module.parameters(),1.0)
        optim.step()
        local_history.append(float(loss.detach()))
    # End-to-end KL/CE: teacher predictions are fixed, other student weights frozen.
    module.eval()
    optim=torch.optim.AdamW(module.parameters(),lr=TRAIN_GLOBAL_LR,weight_decay=0)
    global_history=[]
    for step in range(GLOBAL_DISTILL_STEPS):
        index=step%len(CAL_PAIRS)
        b,gold=CAL_PAIRS[index]
        target=REF_CAL[index].to("cuda")
        optim.zero_grad(set_to_none=True)
        logits=student(**b,use_cache=False).logits[:,-DECISION_POSITIONS:,:].float()
        temperature=1.5
        kd=F.kl_div(
            F.log_softmax(logits/temperature,dim=-1),
            F.softmax(target/temperature,dim=-1),
            reduction="batchmean")*temperature**2/DECISION_POSITIONS
        supervised=F.cross_entropy(
            logits.reshape(-1,logits.shape[-1]),
            gold[:,-DECISION_POSITIONS:].reshape(-1),reduction="mean")
        loss=kd+0.10*supervised
        loss.backward()
        torch.nn.utils.clip_grad_norm_(module.parameters(),1.0)
        optim.step()
        global_history.append(dict(kd=float(kd.detach()),
                                   supervised=float(supervised.detach()),
                                   total=float(loss.detach())))
        del logits,target
    del optim,xx,yy
    return {"local_relative_losses":local_history,
            "global_distillation_losses":global_history}
print("Training starts: candidates",len(TRAIN_ORDER),
      "local steps",LOCAL_STEPS,"global distillation steps",GLOBAL_DISTILL_STEPS)


In [ ]:
#@title 7. Greedy multilevel conversion: train -> packed 2-bit -> DEV/GUARD quality gate
def checkpoint_progress():
    (OUT/"training_progress.json").write_text(json.dumps({
        "source":SOURCE,"source_sha":SOURCE_SHA,
        "selected":SELECTED,"attempts":TRIALS,
        "trained_paths":TRAIN_ORDER,
        "completed_attempts":len(TRIALS)},indent=2))
for index,name in enumerate(TRAIN_ORDER,1):
    if len(SELECTED)>=MAX_ACCEPTED:break
    old=student.get_submodule(name)
    assert isinstance(old,nn.Linear),name
    modes=[0,16] if USE_RANK16_RESCUE else [0]
    accepted=False
    for rank in modes:
        row={"module":name,"rank":rank,"feature_cenn":USE_FEATURE_CENN,
             "accepted":False}
        try:
            qat_module=qat.TernaryQATLinear(old,rank=rank,
                                    feature_residual=USE_FEATURE_CENN).cuda()
            qat.swap(student,name,qat_module)
            history=optimize_candidate(qat_module,name,rank)
            packed=qat_module.pack(CHUNK_ROWS).cuda().eval()
            qat.swap(student,name,packed)
            dev=qat.last_token_metrics(student,DEV,REF_DEV,DECISION_POSITIONS)
            guard=qat.last_token_metrics(student,GUARD,REF_GUARD,DECISION_POSITIONS)
            dev_ppl=qat.perplexity(student,DEV_TEXT)["perplexity"]
            good=qat.is_acceptable(
                dev,guard,DEV_BASE_PPL,dev_ppl,
                max_errors=MAX_DISAGREEMENTS,max_js=MAX_JS,
                max_ppl_increase=MAX_DEV_PPL_INCREASE)
            original_size=qat.projection_bytes(old)
            packed_size=qat.projection_bytes(packed)
            row.update(accepted=good,dev=dev,guard=guard,
                       dev_ppl=dev_ppl,old_state_bytes=original_size,
                       packed_state_bytes=packed_size,
                       relative_packed_saving=1-packed_size/original_size,
                       feature_gain=float(
                         packed.feature_gain.float()) if packed.feature_gain is not None else None,
                       training=history)
            if good:
                metadata=dict(name=name,rank=rank,chunk_rows=CHUNK_ROWS,
                    feature_residual=USE_FEATURE_CENN,
                    checkpoint_bytes=packed_size,
                    baseline_bytes=original_size)
                SELECTED.append(metadata)
                accepted=True
                print("ACCEPT",name,
                      "rank",rank,"dev agreement",round(dev["agreement"],4),
                      "guard agreement",round(guard["agreement"],4),
                      "state saving",round(100*(1-packed_size/original_size),2),"%")
                del qat_module
                break
            qat.swap(student,name,old)
        except Exception as exc:
            qat.swap(student,name,old)
            row["error"]=repr(exc)[:700]
            print("REJECT: runtime failure",name,"rank",rank,row["error"][:200])
        TRIALS.append(row)
        checkpoint_progress()
        if "packed" in locals():del packed
        if "qat_module" in locals():del qat_module
        gc.collect();torch.cuda.empty_cache()
    if accepted:
        TRIALS.append(row)
        checkpoint_progress()
    print("Candidates completed",index,"/",len(TRAIN_ORDER),
          "| packed ternary modules retained:",len(SELECTED))
checkpoint_progress()
print("Final packed replacements",len(SELECTED),
      "model state MiB",qat.tensor_storage_bytes(student)/2**20,
      "baseline",ORIGINAL_STATE/2**20)


In [ ]:
#@title 8. SEALED validation/test: agreement across 16 positions + WikiText gold PPL
# Do not run this cell until the search has finished: validation/test are held out.
REF_AUDIT=ref_for(AUDIT)
REF_TEST=ref_for(TEST)
VALIDATION_AGREEMENT=qat.last_token_metrics(
    student,AUDIT,REF_AUDIT,DECISION_POSITIONS)
TEST_AGREEMENT=qat.last_token_metrics(
    student,TEST,REF_TEST,DECISION_POSITIONS)
GOLD={}
for label,model in (("original_fp16",teacher),("multi_layer_ternary",student)):
    GOLD[label]={
        "validation":qat.perplexity(model,AUDIT_TEXT),
        "test":qat.perplexity(model,TEST_TEXT)}
# PASS requires sufficient out-of-sample quality; no automatic HF publication.
val_ratio=(GOLD["multi_layer_ternary"]["validation"]["perplexity"] /
           GOLD["original_fp16"]["validation"]["perplexity"])
test_ratio=(GOLD["multi_layer_ternary"]["test"]["perplexity"] /
            GOLD["original_fp16"]["test"]["perplexity"])
QUALITY_PASS=bool(SELECTED) and VALIDATION_AGREEMENT["agreement"]>=.99 and (
    val_ratio<=1.02) and (test_ratio<=1.02)
QUALITY={
  "sealed_validation":VALIDATION_AGREEMENT,
  "sealed_test":TEST_AGREEMENT,
  "gold_wikitext":GOLD,
  "validation_perplexity_ratio":val_ratio,
  "test_perplexity_ratio":test_ratio,
  "heldout_quality_pass":QUALITY_PASS,
  "disclaimer":"Held-out test is a final report, NEVER used to choose projections."
}
(OUT/"independent_quality_v11.json").write_text(json.dumps(QUALITY,indent=2))
print("SEALED AUDIT:",json.dumps(QUALITY,indent=2))


In [ ]:
#@title 9. Inference CUDA latency and allocation (teacher & candidate both resident)
@torch.inference_mode()
def paired_latency(batch,repeats=BENCH_REPEATS):
    def run(model):return model(**batch,use_cache=False).logits
    for _ in range(3):run(teacher);run(student)
    torch.cuda.synchronize()
    def clock(model):
        a=torch.cuda.Event(enable_timing=True)
        z=torch.cuda.Event(enable_timing=True)
        a.record()
        logits=run(model)
        z.record();z.synchronize()
        del logits
        return a.elapsed_time(z)
    original=[];optimized=[]
    for i in range(repeats):
        if i%2:
            c1=clock(student);o1=clock(teacher);o2=clock(teacher);c2=clock(student)
        else:
            o1=clock(teacher);c1=clock(student);c2=clock(student);o2=clock(teacher)
        original.append((o1+o2)/2)
        optimized.append((c1+c2)/2)
    ratios=[o/t for o,t in zip(original,optimized)]
    return {"fp16_ms":statistics.median(original),
            "candidate_ms":statistics.median(optimized),
            "paired_speedup":statistics.median(ratios)}
BENCH=[]
if SELECTED:
    for B,T in ((1,64),(1,128),(4,64),(4,128),(4,256)):
        input_ids=torch.randint(0,len(tokenizer),(B,T),device="cuda")
        data={"input_ids":input_ids,
              "attention_mask":torch.ones_like(input_ids)}
        row={"batch":B,"tokens":T,**paired_latency(data)}
        for label,model in (("fp16",teacher),("ternary",student)):
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
            start_mem=torch.cuda.memory_allocated()
            result=model(**data,use_cache=False).logits
            torch.cuda.synchronize()
            row[label+"_forward_extra_peak_mib"]=(
                torch.cuda.max_memory_allocated()-start_mem)/2**20
            del result
        BENCH.append(row)
        print("B",B,"T",T,"paired speedup",round(row["paired_speedup"],3))
else:print("No replacement passed; skip FP16-versus-FP16 pseudo speedup.")
pd.DataFrame(BENCH).to_csv(OUT/"latency_v11.csv",index=False)
if BENCH:display(pd.DataFrame(BENCH))


In [ ]:
#@title 10. Deterministic chat generation and causal-safe tokenizer handling
CHATS=[]
PROMPTS=[
    "Explain in two sentences why the sky is blue.",
    "Calculate 17 multiplied by 23 and give the result."
]
for prompt in PROMPTS:
    item={"prompt":prompt}
    try:
        text_prompt=tokenizer.apply_chat_template(
            [{"role":"user","content":prompt}],
            tokenize=False,add_generation_prompt=True)
        encoded=tokenizer(text_prompt,add_special_tokens=False,return_tensors="pt")
        input_ids=encoded["input_ids"].cuda()
        attention_mask=encoded.get("attention_mask")
        if attention_mask is None:attention_mask=torch.ones_like(encoded["input_ids"])
        attention_mask=attention_mask.cuda()
        for label,model in (("fp16",teacher),("ternary",student)):
            with torch.inference_mode():
                output=model.generate(
                    input_ids=input_ids,attention_mask=attention_mask,
                    max_new_tokens=48,do_sample=False,use_cache=True,
                    pad_token_id=tokenizer.pad_token_id)
            item[label]=tokenizer.decode(output[0,input_ids.shape[1]:],
                                         skip_special_tokens=True)
        print("USER:",prompt,"FP16:",item["fp16"],"TERNARY:",item["ternary"])
    except Exception as exc:
        item["error"]=repr(exc)[:400]
        print("Chat comparison failed (nonfatal):",item["error"])
    CHATS.append(item)
(OUT/"chat_comparison_v11.json").write_text(json.dumps(CHATS,indent=2))


In [ ]:
#@title 11. Save packed 2-bit delta, cold-reload exact state, save evidence ZIP
MANIFEST={
 "name":"SmolLM2 V11 Multilayer Activation-Aware Ternary CeNN QAT",
 "source_repo":SOURCE,"source_revision":SOURCE_SHA,
 "code_v9_revision":V9_SHA,"code_v11_revision":V11_SHA,
 "training_dataset":WIKI_REPO,"dataset_config":WIKI_CONFIG,
 "quantization":"strict signed {-1,0,+1}, 2-bit G128 packed, FP16 scales",
 "residuals":"optional rank16 FP16; not purely ternary if rank > 0",
 "feature_cenn":"causal-safe fixed [-1,+1,0] feature neighborhood and learned gain",
 "active_modules":SELECTED,
 "requires_exact_FP16_base":True,
 "source_model_never_overwritten":True,
 "quantized_inference_backend":"FP16 tiled decompression; not native ternary GEMM",
 "quality_pass":QUALITY_PASS}
(OUT/"packed_delta_manifest_v11.json").write_text(json.dumps(MANIFEST,indent=2))
delta=OUT/"smol360_v11_g128_ternary_cenn_delta.safetensors"
# Do not accidentally archive a prior run's accepted delta after a zero-win run.
if delta.exists():delta.unlink()
RELOAD_SUCCESS=None
if SELECTED:
    tensors={}
    for entry in SELECTED:
        for suffix,value in student.get_submodule(entry["name"]).state_dict().items():
            tensors[entry["name"]+"."+suffix]=value.detach().cpu().contiguous()
    save_file(tensors,str(delta))
    # Strict cold-load into a new original teacher without retraining anything.
    reloaded=copy.deepcopy(teacher).eval().requires_grad_(False)
    checkpoint=load_file(str(delta),device="cpu")
    for entry in SELECTED:
        path=entry["name"]
        initial=reloaded.get_submodule(path)
        replacement=qat.create_from_metadata(initial,entry,device="cuda")
        tensors_for_module={
            key[len(path)+1:]:value.cuda()
            for key,value in checkpoint.items() if key.startswith(path+".")}
        replacement.load_state_dict(tensors_for_module,strict=True)
        qat.swap(reloaded,path,replacement)
    with torch.inference_mode():
        expected=student(**DEV[0],use_cache=False).logits[:,-1,:].float()
        obtained=reloaded(**DEV[0],use_cache=False).logits[:,-1,:].float()
    torch.testing.assert_close(expected,obtained,rtol=.005,atol=.005)
    RELOAD_SUCCESS=True
    print("Packed delta reloaded with matching forward output:",
          delta.stat().st_size/2**20,"MiB")
    del reloaded,checkpoint,tensors,expected,obtained
    gc.collect();torch.cuda.empty_cache()
else:print("No accepted layers; do not export a dummy compressed checkpoint.")
FINAL={
 "experiment":"SmolLM2 V11 Ternary CeNN QAT with local+global teacher distillation",
 "source":SOURCE,"source_sha":SOURCE_SHA,
 "gpu":torch.cuda.get_device_name(0),
 "candidates_screened":len(SCREEN),
 "candidates_trained":TRAIN_ORDER,
 "attempts":TRIALS,"selected":SELECTED,
 "selected_count":len(SELECTED),
 "original_model_state_mib":ORIGINAL_STATE/2**20,
 "candidate_model_state_mib":qat.tensor_storage_bytes(student)/2**20,
 "state_saving_pct":100*(1-qat.tensor_storage_bytes(student)/ORIGINAL_STATE),
 "heldout_validation":QUALITY,
 "latency":BENCH,"chat":CHATS,
 "delta_size_mib":delta.stat().st_size/2**20 if delta.exists() else None,
 "delta_reload_pass":RELOAD_SUCCESS,
 "pilot_quality_and_reload_gate_pass":bool(QUALITY_PASS and RELOAD_SUCCESS),
 "model_eligible_for_publication":False,
 "caveats":[
   "Calibration/dev/guard are disjoint WikiText training rows, validation/test held out.",
   "No sparse or dynamic recurrent CeNN across token time is claimed.",
   "QAT master FP32 weights discarded during packed export.",
   "Inference uses tile reconstruction, so there is no native low-bit speedup claim.",
   "Do not upload without broader independent tests and privacy-safe model review."]}
(OUT/"v11_full_report.json").write_text(json.dumps(FINAL,indent=2))
zipout=OUT/"SmolLM2_360M_V11_Ternary_CeNN_QAT_Results.zip"
with zipfile.ZipFile(zipout,"w",zipfile.ZIP_DEFLATED) as archive:
    for file in OUT.iterdir():
        if file.is_file() and file!=zipout:archive.write(file,file.name)
print("Full results:",zipout)
try:
    from google.colab import files
    files.download(str(zipout))
except ImportError:pass


## How to interpret V11

**Success is conditional.** A candidate is accepted only after full-model *packed* evaluation on disjoint WikiText train development and guard rows; validation/test metrics are computed only after the search completes. The exported checkpoint can contain zero or multiple ternary replacements.

**Strict ternary versus hybrid:** G128 code values are exactly −1,0,+1 and use 2-bit packing; optional rank-16 FP16 residual corrects quantization error but is a hybrid, not a pure ternary layer. A trainable feature-neighborhood CeNN gain is reported separately; no lookahead-token recurrence is introduced.

**Memory versus performance:** Percent reduction of full-model state is different from GPU forward peak allocations. The tile decoder is expected to be slower on a T4 unless kernel development follows. Compare quality and both speed directions before any claim.

**Research limitations:** Six trained candidate sites and 48 windows per held-out split are still pilot-scale, not publication-level statistics. A strong result should be rerun with larger corpora, multiple seeds, actual decoding throughput and an independently tested loader/quantized CUDA kernel before HF publication.
